<a href="https://colab.research.google.com/github/krishnasivaprasadm-jpg/cardiovascular_CA-SAE-AFB-a/blob/main/CA_SAE_AFB_Corrected_Ablation_T4_Complexity(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CA-SAE-AFB
# CORRECTED REVIEWER-READY ABLATION EXPERIMENT
# ============================================================
#
# Purpose:
# 1. Strict OOF stacking
# 2. Leakage-controlled threshold optimization
# 3. Independent component ablations
# 4. Same outer folds for every configuration
# 5. T4 GPU latency measurement
# 6. Parameter count
# 7. Big-O complexity analysis
#
# Dataset:
#     heart.csv
#
# Target:
#     HeartDisease
#
# IMPORTANT:
# Run the notebook from beginning to end on Google Colab T4.
# Do NOT mix these results with the previous ablation results.
# ============================================================


# ============================================================
# CELL 1 — INSTALLATION
# ============================================================

!pip -q install xgboost openpyxl


# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import os
import gc
import time
import random
import json
import platform

import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score
)

from scipy.stats import ttest_rel, wilcoxon, t

print("TensorFlow:", tf.__version__)
print("XGBoost:", xgb.__version__)
print("Python:", platform.python_version())


# ============================================================
# CELL 3 — REPRODUCIBILITY
# ============================================================

SEED = 2026

OUTER_FOLDS = 5
INNER_FOLDS = 3

MAX_EPOCHS = 200
BATCH_SIZE = 32
PATIENCE = 10


def seed_all(seed):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)

    np.random.seed(seed)

    tf.random.set_seed(seed)


seed_all(SEED)

print("Seed:", SEED)


# ============================================================
# CELL 4 — GPU / T4 INFORMATION
# ============================================================

print("Available GPU devices:")
print(tf.config.list_physical_devices("GPU"))

print("\nGPU information:")
!nvidia-smi

print("\nT4-specific information:")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

print("\nCPU information:")
!lscpu | head -20

print("\nRAM information:")
!free -h


# ============================================================
# CELL 5 — UPLOAD DATASET
# ============================================================

from google.colab import files

uploaded = files.upload()

if "heart.csv" in uploaded:

    filename = "heart.csv"

else:

    filename = list(uploaded.keys())[0]

print("Using file:", filename)


# ============================================================
# CELL 6 — LOAD DATA
# ============================================================

df = pd.read_csv(filename)

TARGET = "HeartDisease"

if TARGET not in df.columns:

    raise ValueError(
        f"Target '{TARGET}' not found.\n"
        f"Available columns:\n{list(df.columns)}"
    )

X_df = df.drop(columns=[TARGET]).copy()

y = df[TARGET].astype(int).to_numpy()


# One-hot encode categorical variables.
X_df = pd.get_dummies(
    X_df,
    drop_first=True
)


# Replace infinity with NaN.
# Imputation is performed later inside each fold.
X_df = X_df.replace(
    [np.inf, -np.inf],
    np.nan
)


print("Dataset shape:", X_df.shape)

print("\nTarget distribution:")
print(
    pd.Series(y).value_counts()
)

print("\nPredictor columns:")
print(X_df.columns.tolist())


# ============================================================
# CELL 7 — PREPROCESSING FUNCTION
# ============================================================
#
# IMPORTANT:
#
# Imputer:
#     fit ONLY on training fold.
#
# PolynomialFeatures:
#     fit ONLY on training fold.
#
# Scaler:
#     fit ONLY on training fold.
#
# Therefore validation/test data never influence
# preprocessing parameters.
# ============================================================

def preprocess_fold(
    X_train,
    X_validation,
    X_test,
    use_interactions=True
):

    # --------------------------------------------------------
    # Missing-value imputation
    # --------------------------------------------------------

    imputer = SimpleImputer(
        strategy="median"
    )

    X_train_imp = imputer.fit_transform(
        X_train
    )

    X_validation_imp = imputer.transform(
        X_validation
    )

    X_test_imp = imputer.transform(
        X_test
    )


    # --------------------------------------------------------
    # Interaction features
    # --------------------------------------------------------

    polynomial = None

    if use_interactions:

        polynomial = PolynomialFeatures(
            degree=2,
            interaction_only=True,
            include_bias=False
        )

        X_train_imp = polynomial.fit_transform(
            X_train_imp
        )

        X_validation_imp = polynomial.transform(
            X_validation_imp
        )

        X_test_imp = polynomial.transform(
            X_test_imp
        )


    # --------------------------------------------------------
    # Standardization
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train_imp
    )

    X_validation_scaled = scaler.transform(
        X_validation_imp
    )

    X_test_scaled = scaler.transform(
        X_test_imp
    )


    return (
        X_train_scaled.astype(np.float32),
        X_validation_scaled.astype(np.float32),
        X_test_scaled.astype(np.float32),
        imputer,
        polynomial,
        scaler
    )


# ============================================================
# CELL 8 — SAE + AFB MODEL
# ============================================================

def build_neural_model(
    input_dim,
    use_sae=True,
    use_afb=True
):

    inputs = layers.Input(
        shape=(input_dim,),
        name="input"
    )


    # ========================================================
    # NO-SAE configuration
    # ========================================================

    if not use_sae:

        classification = layers.Dense(
            1,
            activation="sigmoid",
            name="direct_classification"
        )(inputs)

        model = Model(
            inputs,
            classification
        )

        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=0.0005
            ),
            loss="binary_crossentropy"
        )

        return model, None


    # ========================================================
    # SAE encoder
    # ========================================================

    hidden = layers.Dense(
        128,
        activation="relu",
        activity_regularizer=
            tf.keras.regularizers.l1(1e-4),
        name="encoder_hidden"
    )(inputs)


    hidden = layers.Dropout(
        0.30,
        name="encoder_dropout"
    )(hidden)


    latent = layers.Dense(
        64,
        activation="relu",
        name="latent"
    )(hidden)


    # ========================================================
    # AFB
    # ========================================================

    if use_afb:

        weights = layers.Dense(
            64,
            activation="sigmoid",
            name="afb_weight"
        )(latent)

        refined_latent = layers.Multiply(
            name="afb_refined_latent"
        )(
            [latent, weights]
        )

    else:

        refined_latent = latent


    # ========================================================
    # Decoder
    # ========================================================

    reconstruction = layers.Dense(
        input_dim,
        activation="linear",
        name="reconstruction"
    )(refined_latent)


    # ========================================================
    # Classifier
    # ========================================================

    classification = layers.Dense(
        1,
        activation="sigmoid",
        name="classification"
    )(refined_latent)


    # ========================================================
    # Complete SAE
    # ========================================================

    model = Model(
        inputs,
        [
            reconstruction,
            classification
        ]
    )


    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.0005
        ),
        loss=[
            "mse",
            "binary_crossentropy"
        ],
        loss_weights=[
            0.5,
            1.0
        ]
    )


    # Encoder
    encoder = Model(
        inputs,
        refined_latent
    )


    return model, encoder


# ============================================================
# CELL 9 — TRAIN SAE / DIRECT MODEL
# ============================================================

def train_neural_component(
    X_train,
    y_train,
    X_validation,
    y_validation,
    use_sae,
    use_afb
):

    model, encoder = build_neural_model(
        input_dim=X_train.shape[1],
        use_sae=use_sae,
        use_afb=use_afb
    )


    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True
    )


    # ========================================================
    # SAE training
    # ========================================================

    if use_sae:

        model.fit(
            X_train,
            [
                X_train,
                y_train
            ],
            validation_data=(
                X_validation,
                [
                    X_validation,
                    y_validation
                ]
            ),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[
                early_stopping
            ],
            verbose=0
        )


    # ========================================================
    # Direct classifier for NO-SAE
    # ========================================================

    else:

        model.fit(
            X_train,
            y_train,
            validation_data=(
                X_validation,
                y_validation
            ),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[
                early_stopping
            ],
            verbose=0
        )


    return model, encoder


# ============================================================
# CELL 10 — XGBOOST
# ============================================================

def train_xgboost(
    X_train,
    y_train,
    seed
):

    model = xgb.XGBClassifier(

        n_estimators=250,

        max_depth=3,

        learning_rate=0.04,

        subsample=0.85,

        colsample_bytree=0.85,

        reg_lambda=2.0,

        reg_alpha=0.1,

        eval_metric="logloss",

        random_state=seed,

        n_jobs=-1
    )


    model.fit(
        X_train,
        y_train
    )


    return model


# ============================================================
# CELL 11 — LOGISTIC REGRESSION
# ============================================================

def train_logistic_regression(
    X_train,
    y_train,
    seed
):

    model = LogisticRegression(

        max_iter=2000,

        class_weight="balanced",

        random_state=seed
    )


    model.fit(
        X_train,
        y_train
    )


    return model


# ============================================================
# CELL 12 — THRESHOLD OPTIMIZATION
# ============================================================
#
# Threshold is optimized ONLY on OOF predictions.
#
# NEVER use outer test predictions here.
# ============================================================

def optimize_threshold(
    y_true,
    probabilities
):

    threshold_grid = np.arange(
        0.10,
        0.91,
        0.005
    )


    f1_scores = []


    for threshold in threshold_grid:

        prediction = (
            probabilities >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            prediction
        )

        f1_scores.append(
            score
        )


    best_index = int(
        np.argmax(
            f1_scores
        )
    )


    return float(
        threshold_grid[
            best_index
        ]
    )


# ============================================================
# CELL 13 — METRICS
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold
):

    prediction = (
        probabilities >= threshold
    ).astype(int)


    return {

        "Accuracy":
            accuracy_score(
                y_true,
                prediction
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                prediction
            ),

        "Precision":
            precision_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "MCC":
            matthews_corrcoef(
                y_true,
                prediction
            ),

        "ROC_AUC":
            roc_auc_score(
                y_true,
                probabilities
            ),

        "PR_AUC":
            average_precision_score(
                y_true,
                probabilities
            )
    }


# ============================================================
# CELL 14 — CONFIGURATIONS
# ============================================================

CONFIGS = {

    "FULL": {

        "interaction": True,

        "sae": True,

        "afb": True,

        "lr": True,

        "meta": True,

        "threshold": True,

        "two_stage": True
    },


    "NO_INTERACTION": {

        "interaction": False,

        "sae": True,

        "afb": True,

        "lr": True,

        "meta": True,

        "threshold": True,

        "two_stage": True
    },


    "NO_SAE": {

        "interaction": True,

        "sae": False,

        "afb": False,

        "lr": True,

        "meta": True,

        "threshold": True,

        "two_stage": True
    },


    "NO_AFB": {

        "interaction": True,

        "sae": True,

        "afb": False,

        "lr": True,

        "meta": True,

        "threshold": True,

        "two_stage": True
    },


    "NO_LR": {

        "interaction": True,

        "sae": True,

        "afb": True,

        "lr": False,

        "meta": True,

        "threshold": True,

        "two_stage": True
    },


    "NO_META": {

        "interaction": True,

        "sae": True,

        "afb": True,

        "lr": True,

        "meta": False,

        "threshold": True,

        "two_stage": True
    },


    "NO_THRESHOLD": {

        "interaction": True,

        "sae": True,

        "afb": True,

        "lr": True,

        "meta": True,

        "threshold": False,

        "two_stage": True
    },


    "NO_2STAGE": {

        "interaction": True,

        "sae": True,

        "afb": True,

        "lr": True,

        "meta": True,

        "threshold": True,

        "two_stage": False
    }
}


# ============================================================
# CELL 15 — STRICT OOF BASE PREDICTIONS
# ============================================================
#
# This is the critical correction.
#
# The meta learner NEVER receives predictions generated from
# models trained on the same observations.
#
# Every OOF prediction is generated from a model that did NOT
# see that observation during training.
# ============================================================

def generate_oof_predictions(

    X_outer_train,

    y_outer_train,

    config,

    seed

):

    inner_cv = StratifiedKFold(

        n_splits=INNER_FOLDS,

        shuffle=True,

        random_state=seed
    )


    oof_lr = np.zeros(
        len(y_outer_train)
    )


    oof_xgb = np.zeros(
        len(y_outer_train)
    )


    for inner_fold, (
        train_index,
        validation_index
    ) in enumerate(

        inner_cv.split(
            X_outer_train,
            y_outer_train
        ),

        start=1
    ):

        print(
            "   Inner fold:",
            inner_fold
        )


        X_train_raw = (
            X_outer_train.iloc[
                train_index
            ]
        )

        X_validation_raw = (
            X_outer_train.iloc[
                validation_index
            ]
        )


        y_train = (
            y_outer_train[
                train_index
            ]
        )

        y_validation = (
            y_outer_train[
                validation_index
            ]
        )


        # ----------------------------------------------------
        # Fold-local preprocessing
        # ----------------------------------------------------

        (
            X_train,
            X_validation,
            _,
            _,
            _,
            _
        ) = preprocess_fold(

            X_train_raw,

            X_validation_raw,

            X_validation_raw,

            use_interactions=
                config["interaction"]
        )


        # ----------------------------------------------------
        # SAE / direct model
        # ----------------------------------------------------

        neural_model, encoder = (
            train_neural_component(

                X_train,

                y_train,

                X_validation,

                y_validation,

                use_sae=
                    config["sae"],

                use_afb=
                    config["afb"]
            )
        )


        # ----------------------------------------------------
        # XGBoost
        # ----------------------------------------------------

        xgb_model = train_xgboost(

            X_train,

            y_train,

            seed + inner_fold
        )


        oof_xgb[
            validation_index
        ] = xgb_model.predict_proba(
            X_validation
        )[:,1]


        # ----------------------------------------------------
        # LR
        # ----------------------------------------------------

        if config["lr"]:

            if config["sae"]:

                Z_train = (
                    encoder.predict(
                        X_train,
                        verbose=0
                    )
                )

                Z_validation = (
                    encoder.predict(
                        X_validation,
                        verbose=0
                    )
                )

            else:

                Z_train = X_train

                Z_validation = X_validation


            lr_model = (
                train_logistic_regression(

                    Z_train,

                    y_train,

                    seed + inner_fold
                )
            )


            oof_lr[
                validation_index
            ] = lr_model.predict_proba(
                Z_validation
            )[:,1]


        else:

            oof_lr[
                validation_index
            ] = 0.5


        # ----------------------------------------------------
        # Memory cleanup
        # ----------------------------------------------------

        del neural_model

        del encoder

        del xgb_model

        gc.collect()

        tf.keras.backend.clear_session()


    return (
        oof_lr,
        oof_xgb
    )


# ============================================================
# CELL 16 — TRAIN FINAL OUTER-TRAINING MODELS
# ============================================================

def train_outer_models(

    X_outer_train,

    y_outer_train,

    X_outer_test,

    config,

    seed

):

    (
        X_train,
        X_dummy,
        X_test,
        imputer,
        polynomial,
        scaler
    ) = preprocess_fold(

        X_outer_train,

        X_outer_train,

        X_outer_test,

        use_interactions=
            config["interaction"]
    )


    # --------------------------------------------------------
    # SAE
    # --------------------------------------------------------

    neural_model, encoder = (
        train_neural_component(

            X_train,

            y_outer_train,

            X_dummy,

            y_outer_train,

            use_sae=
                config["sae"],

            use_afb=
                config["afb"]
        )
    )


    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    xgb_model = train_xgboost(

        X_train,

        y_outer_train,

        seed
    )


    p_xgb_test = (
        xgb_model.predict_proba(
            X_test
        )[:,1]
    )


    # --------------------------------------------------------
    # LR
    # --------------------------------------------------------

    if config["lr"]:

        if config["sae"]:

            Z_train = (
                encoder.predict(
                    X_train,
                    verbose=0
                )
            )

            Z_test = (
                encoder.predict(
                    X_test,
                    verbose=0
                )
            )

        else:

            Z_train = X_train

            Z_test = X_test


        lr_model = (
            train_logistic_regression(

                Z_train,

                y_outer_train,

                seed
            )
        )


        p_lr_test = (
            lr_model.predict_proba(
                Z_test
            )[:,1]
        )


    else:

        p_lr_test = np.zeros(
            len(X_test)
        )


    return (

        p_lr_test,

        p_xgb_test,

        neural_model,

        encoder,

        xgb_model
    )


# ============================================================
# CELL 17 — NO-META COMBINATION
# ============================================================
#
# NO-META retains both base learners.
#
# Their combination weight is selected using OOF predictions.
#
# This makes NO-META a valid removal of the Gradient Boosting
# meta learner rather than accidentally converting the model
# into XGBoost-only.
# ============================================================

def select_base_weight(

    y_train,

    oof_lr,

    oof_xgb

):

    weight_grid = np.arange(
        0.0,
        1.01,
        0.01
    )


    best_weight = 0.5

    best_score = -np.inf


    for weight in weight_grid:

        probability = (

            weight * oof_lr

            +

            (1-weight) * oof_xgb

        )


        threshold = (
            optimize_threshold(
                y_train,
                probability
            )
        )


        prediction = (
            probability >= threshold
        ).astype(int)


        score = f1_score(
            y_train,
            prediction
        )


        if score > best_score:

            best_score = score

            best_weight = weight


    return best_weight


# ============================================================
# CELL 18 — TWO-STAGE DECISION
# ============================================================

def two_stage_prediction(

    probability,

    threshold

):

    margin = 0.05


    lower = max(
        0.0,
        threshold - margin
    )


    upper = min(
        1.0,
        threshold + margin
    )


    prediction = np.where(

        probability >= upper,

        1,

        np.where(

            probability <= lower,

            0,

            (
                probability >= threshold
            ).astype(int)
        )
    )


    return prediction


# ============================================================
# CELL 19 — RUN ONE CONFIGURATION
# ============================================================

def run_configuration(

    config_name,

    config,

    X,

    y,

    seed=SEED

):

    outer_cv = StratifiedKFold(

        n_splits=OUTER_FOLDS,

        shuffle=True,

        random_state=seed
    )


    fold_results = []


    for fold, (

        train_index,

        test_index

    ) in enumerate(

        outer_cv.split(
            X,
            y
        ),

        start=1
    ):

        print(
            "\n================================================"
        )

        print(
            config_name,
            "Outer Fold",
            fold
        )

        print(
            "================================================"
        )


        X_outer_train = (
            X.iloc[
                train_index
            ]
        )


        X_outer_test = (
            X.iloc[
                test_index
            ]
        )


        y_outer_train = (
            y[
                train_index
            ]
        )


        y_outer_test = (
            y[
                test_index
            ]
        )


        fold_seed = (
            seed + fold
        )


        # ====================================================
        # STEP 1
        # STRICT OOF PREDICTIONS
        # ====================================================

        (
            oof_lr,

            oof_xgb

        ) = generate_oof_predictions(

            X_outer_train,

            y_outer_train,

            config,

            fold_seed
        )


        # ====================================================
        # STEP 2
        # META LEARNER
        # ====================================================

        if config["meta"]:

            meta_model = (
                GradientBoostingClassifier(

                    n_estimators=200,

                    learning_rate=0.05,

                    max_depth=2,

                    random_state=fold_seed
                )
            )


            meta_model.fit(

                np.column_stack(
                    [
                        oof_lr,
                        oof_xgb
                    ]
                ),

                y_outer_train
            )


            p_oof = (
                meta_model.predict_proba(

                    np.column_stack(
                        [
                            oof_lr,
                            oof_xgb
                        ]
                    )

                )[:,1]
            )


        else:

            # NO-META
            #
            # Both LR and XGB remain.
            #
            # No Gradient Boosting meta learner.

            selected_weight = (
                select_base_weight(

                    y_outer_train,

                    oof_lr,

                    oof_xgb
                )
            )


            p_oof = (

                selected_weight * oof_lr

                +

                (1-selected_weight) * oof_xgb
            )


        # ====================================================
        # STEP 3
        # THRESHOLD
        # ====================================================

        if config["threshold"]:

            threshold = (
                optimize_threshold(

                    y_outer_train,

                    p_oof
                )
            )

        else:

            threshold = 0.50


        print(
            "Selected threshold:",
            threshold
        )


        # ====================================================
        # STEP 4
        # FINAL OUTER TRAINING
        # ====================================================

        (
            p_lr_test,

            p_xgb_test,

            neural_model,

            encoder,

            xgb_model

        ) = train_outer_models(

            X_outer_train,

            y_outer_train,

            X_outer_test,

            config,

            fold_seed
        )


        # ====================================================
        # STEP 5
        # OUTER TEST PREDICTION
        # ====================================================

        if config["meta"]:

            p_test = (
                meta_model.predict_proba(

                    np.column_stack(
                        [
                            p_lr_test,
                            p_xgb_test
                        ]
                    )

                )[:,1]
            )


        else:

            p_test = (

                selected_weight * p_lr_test

                +

                (1-selected_weight)
                * p_xgb_test
            )


        # ====================================================
        # STEP 6
        # CLASSIFICATION DECISION
        # ====================================================

        if config["two_stage"]:

            prediction = (
                two_stage_prediction(

                    p_test,

                    threshold
                )
            )

        else:

            prediction = (

                p_test >= threshold
            ).astype(int)


        # ====================================================
        # STEP 7
        # METRICS
        # ====================================================

        metrics = {

            "Accuracy":
                accuracy_score(
                    y_outer_test,
                    prediction
                ),

            "Balanced_Accuracy":
                balanced_accuracy_score(
                    y_outer_test,
                    prediction
                ),

            "Precision":
                precision_score(
                    y_outer_test,
                    prediction,
                    zero_division=0
                ),

            "Recall":
                recall_score(
                    y_outer_test,
                    prediction,
                    zero_division=0
                ),

            "F1":
                f1_score(
                    y_outer_test,
                    prediction,
                    zero_division=0
                ),

            "MCC":
                matthews_corrcoef(
                    y_outer_test,
                    prediction
                ),

            "ROC_AUC":
                roc_auc_score(
                    y_outer_test,
                    p_test
                ),

            "PR_AUC":
                average_precision_score(
                    y_outer_test,
                    p_test
                )
        }


        metrics.update({

            "Configuration":
                config_name,

            "Fold":
                fold,

            "Threshold":
                threshold
        })


        fold_results.append(
            metrics
        )


        print(
            "Accuracy:",
            metrics["Accuracy"]
        )

        print(
            "F1:",
            metrics["F1"]
        )

        print(
            "MCC:",
            metrics["MCC"]
        )


        # ====================================================
        # CLEANUP
        # ====================================================

        del neural_model

        del encoder

        del xgb_model

        gc.collect()

        tf.keras.backend.clear_session()


    return pd.DataFrame(
        fold_results
    )


# ============================================================
# CELL 20 — RUN ALL ABLATIONS
# ============================================================

all_results = []


for (

    config_name,

    config

) in CONFIGS.items():

    result = run_configuration(

        config_name,

        config,

        X_df,

        y,

        seed=SEED
    )


    all_results.append(
        result
    )


ablation_results = pd.concat(
    all_results,
    ignore_index=True
)


ablation_results.to_csv(

    "corrected_ablation_outer5_results.csv",

    index=False
)


print(
    "\nSaved:"
)

print(
    "corrected_ablation_outer5_results.csv"
)


# ============================================================
# CELL 21 — SUMMARY
# ============================================================

summary = (
    ablation_results
    .groupby("Configuration")
    .agg(

        Accuracy_mean=(
            "Accuracy",
            "mean"
        ),

        Accuracy_sd=(
            "Accuracy",
            "std"
        ),

        Balanced_Accuracy_mean=(
            "Balanced_Accuracy",
            "mean"
        ),

        Balanced_Accuracy_sd=(
            "Balanced_Accuracy",
            "std"
        ),

        Precision_mean=(
            "Precision",
            "mean"
        ),

        Precision_sd=(
            "Precision",
            "std"
        ),

        Recall_mean=(
            "Recall",
            "mean"
        ),

        Recall_sd=(
            "Recall",
            "std"
        ),

        F1_mean=(
            "F1",
            "mean"
        ),

        F1_sd=(
            "F1",
            "std"
        ),

        MCC_mean=(
            "MCC",
            "mean"
        ),

        MCC_sd=(
            "MCC",
            "std"
        ),

        ROC_AUC_mean=(
            "ROC_AUC",
            "mean"
        ),

        ROC_AUC_sd=(
            "ROC_AUC",
            "std"
        ),

        PR_AUC_mean=(
            "PR_AUC",
            "mean"
        ),

        PR_AUC_sd=(
            "PR_AUC",
            "std"
        )
    )
    .reset_index()
)


summary.to_csv(

    "corrected_ablation_outer5_summary.csv",

    index=False
)


display(
    summary
)


# ============================================================
# CELL 22 — ABLATION STATISTICAL TESTS
# ============================================================

full = (
    ablation_results[
        ablation_results["Configuration"]
        == "FULL"
    ]
    .sort_values("Fold")
)


statistical_results = []


for config_name in CONFIGS:

    if config_name == "FULL":

        continue


    other = (
        ablation_results[
            ablation_results["Configuration"]
            == config_name
        ]
        .sort_values("Fold")
    )


    for metric in [

        "Accuracy",

        "F1",

        "MCC",

        "ROC_AUC",

        "PR_AUC"

    ]:

        full_values = (
            full[
                metric
            ].to_numpy()
        )


        other_values = (
            other[
                metric
            ].to_numpy()
        )


        t_stat, t_p = (
            ttest_rel(

                full_values,

                other_values
            )
        )


        try:

            w_stat, w_p = (
                wilcoxon(

                    full_values,

                    other_values
                )
            )

        except Exception:

            w_stat = np.nan

            w_p = np.nan


        statistical_results.append({

            "Ablation":
                config_name,

            "Metric":
                metric,

            "FULL_mean":
                full_values.mean(),

            "Ablation_mean":
                other_values.mean(),

            "Difference_FULL_minus_Ablation":
                (
                    full_values.mean()
                    -
                    other_values.mean()
                ),

            "Paired_t_p":
                t_p,

            "Wilcoxon_p":
                w_p
        })


ablation_statistics = pd.DataFrame(
    statistical_results
)


ablation_statistics.to_csv(

    "corrected_ablation_statistical_tests.csv",

    index=False
)


display(
    ablation_statistics
)


# ============================================================
# CELL 23 — T4 LATENCY FUNCTION
# ============================================================

def synchronize_gpu():

    try:

        tf.config.experimental.async_wait()

    except Exception:

        pass


def benchmark_latency(

    model,

    X_sample,

    repeats=30,

    warmup=10

):

    X_sample = np.asarray(

        X_sample,

        dtype=np.float32
    )


    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    for _ in range(
        warmup
    ):

        model.predict(

            X_sample,

            verbose=0
        )


    synchronize_gpu()


    measurements = []


    # --------------------------------------------------------
    # Timing
    # --------------------------------------------------------

    for _ in range(
        repeats
    ):

        start = (
            time.perf_counter()
        )


        model.predict(

            X_sample,

            verbose=0
        )


        synchronize_gpu()


        elapsed = (

            time.perf_counter()
            -
            start

        ) * 1000


        milliseconds_per_sample = (

            elapsed
            /
            len(X_sample)
        )


        measurements.append(

            milliseconds_per_sample
        )


    measurements = np.asarray(
        measurements
    )


    mean_value = (
        measurements.mean()
    )


    standard_deviation = (
        measurements.std(
            ddof=1
        )
    )


    confidence_interval = (

        t.ppf(

            0.975,

            len(measurements)-1

        )

        *

        standard_deviation

        /

        np.sqrt(
            len(measurements)
        )
    )


    return {

        "Mean_ms_per_sample":
            mean_value,

        "Median_ms_per_sample":
            np.median(
                measurements
            ),

        "SD_ms_per_sample":
            standard_deviation,

        "95CI_Low":
            mean_value
            -
            confidence_interval,

        "95CI_High":
            mean_value
            +
            confidence_interval,

        "Minimum_ms_per_sample":
            measurements.min(),

        "Maximum_ms_per_sample":
            measurements.max(),

        "Repeats":
            repeats,

        "Warmup":
            warmup,

        "Batch_Size":
            len(X_sample)
    }


# ============================================================
# CELL 24 — BUILD T4 BENCHMARK MODEL
# ============================================================

INPUT_DIM = 120


latency_model, latency_encoder = (
    build_neural_model(

        INPUT_DIM,

        use_sae=True,

        use_afb=True
    )
)


dummy_input = (
    np.random.randn(
        1,
        INPUT_DIM
    ).astype(
        np.float32
    )
)


# Initialize model
latency_model.predict(
    dummy_input,
    verbose=0
)


# ============================================================
# CELL 25 — T4 LATENCY
# ============================================================

latency_result = (
    benchmark_latency(

        latency_encoder,

        dummy_input,

        repeats=30,

        warmup=10
    )
)


print(
    "\nT4 latency result:"
)

for key, value in (
    latency_result.items()
):

    print(
        key,
        ":",
        value
    )


# ============================================================
# CELL 26 — MODEL PARAMETERS AND SIZE
# ============================================================

parameter_count = (
    latency_model.count_params()
)


latency_model.save(

    "CA_SAE_AFB_neural_component.keras",

    include_optimizer=False
)


model_size_mb = (

    os.path.getsize(

        "CA_SAE_AFB_neural_component.keras"

    )

    /

    (1024**2)
)


print(
    "Parameter count:",
    parameter_count
)


print(
    "Serialized model size (MB):",
    model_size_mb
)


# ============================================================
# CELL 27 — PARAMETER BREAKDOWN
# ============================================================

p_original = 15

p_expanded = (

    p_original
    +
    (
        p_original
        *
        (p_original-1)
        //
        2
    )
)


hidden = 128

latent = 64


parameter_breakdown = pd.DataFrame({

    "Component": [

        "Encoder 120 -> 128",

        "Latent 128 -> 64",

        "AFB 64 -> 64",

        "Decoder 64 -> 120",

        "Classifier 64 -> 1"

    ],

    "Parameters": [

        p_expanded * hidden
        +
        hidden,

        hidden * latent
        +
        latent,

        latent * latent
        +
        latent,

        latent * p_expanded
        +
        p_expanded,

        latent + 1
    ]
})


total_parameters = (
    parameter_breakdown[
        "Parameters"
    ].sum()
)


print(
    parameter_breakdown
)


print(
    "\nTotal analytical neural parameters:",
    total_parameters
)


# ============================================================
# CELL 28 — BIG-O COMPLEXITY
# ============================================================

complexity_table = pd.DataFrame({

    "Stage": [

        "Interaction expansion",

        "SAE encoder",

        "AFB",

        "Decoder",

        "Classifier",

        "XGBoost training",

        "XGBoost inference",

        "Gradient Boosting meta training",

        "Gradient Boosting meta inference"

    ],


    "Complexity": [

        "O(n p^2)",

        "O(n(p*128 + 128*64))",

        "O(n*64^2)",

        "O(n*64*p)",

        "O(n*64)",

        "Approx. O(T*n*p*log n)",

        "O(T*D)",

        "Approx. O(M*n*log n)",

        "O(M*D)"

    ],


    "Description": [

        "Degree-2 interaction-only feature generation",

        "Dense encoder forward computation",

        "Adaptive 64-dimensional weighting",

        "Latent reconstruction",

        "Binary classification output",

        "Tree construction; implementation dependent",

        "Tree traversal",

        "Meta-ensemble tree construction",

        "Meta-tree traversal"

    ]
})


display(
    complexity_table
)


# ============================================================
# CELL 29 — SAVE COMPLEXITY
# ============================================================

parameter_breakdown.to_csv(

    "corrected_parameter_breakdown.csv",

    index=False
)


complexity_table.to_csv(

    "corrected_complexity_table.csv",

    index=False
)


# ============================================================
# CELL 30 — SAVE LATENCY
# ============================================================

latency_output = pd.DataFrame(
    [latency_result]
)


latency_output[
    "Model_Size_MB"
] = model_size_mb


latency_output[
    "Parameter_Count"
] = parameter_count


latency_output.to_csv(

    "T4_latency_results.csv",

    index=False
)


# ============================================================
# CELL 31 — COMBINED EXCEL EVIDENCE FILE
# ============================================================

with pd.ExcelWriter(

    "CA_SAE_AFB_Corrected_Evidence.xlsx",

    engine="openpyxl"

) as writer:


    ablation_results.to_excel(

        writer,

        index=False,

        sheet_name="Ablation_Folds"
    )


    summary.to_excel(

        writer,

        index=False,

        sheet_name="Ablation_Summary"
    )


    ablation_statistics.to_excel(

        writer,

        index=False,

        sheet_name="Ablation_Statistics"
    )


    parameter_breakdown.to_excel(

        writer,

        index=False,

        sheet_name="Parameters"
    )


    complexity_table.to_excel(

        writer,

        index=False,

        sheet_name="Complexity"
    )


    latency_output.to_excel(

        writer,

        index=False,

        sheet_name="T4_Latency"
    )


print(
    "\n============================================"
)

print(
    "ALL EVIDENCE FILES GENERATED"
)

print(
    "============================================"
)

print(
    "1. corrected_ablation_outer5_results.csv"
)

print(
    "2. corrected_ablation_outer5_summary.csv"
)

print(
    "3. corrected_ablation_statistical_tests.csv"
)

print(
    "4. corrected_parameter_breakdown.csv"
)

print(
    "5. corrected_complexity_table.csv"
)

print(
    "6. T4_latency_results.csv"
)

print(
    "7. CA_SAE_AFB_Corrected_Evidence.xlsx"
)

TensorFlow: 2.20.0
XGBoost: 3.3.0
Python: 3.12.13
Seed: 2026
Available GPU devices:
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

GPU information:
Mon Aug 17 05:32:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       3MiB /  15360

Saving heart.csv to heart (1).csv
Using file: heart (1).csv
Dataset shape: (918, 15)

Target distribution:
1    508
0    410
Name: count, dtype: int64

Predictor columns:
['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'Sex_M', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA', 'RestingECG_Normal', 'RestingECG_ST', 'ExerciseAngina_Y', 'ST_Slope_Flat', 'ST_Slope_Up']

FULL Outer Fold 1
   Inner fold: 1
   Inner fold: 2
   Inner fold: 3
Selected threshold: 0.40000000000000024
Accuracy: 0.842391304347826
F1: 0.8663594470046083
MCC: 0.6832233875927683

FULL Outer Fold 2
   Inner fold: 1
   Inner fold: 2
   Inner fold: 3
Selected threshold: 0.5250000000000004
Accuracy: 0.875
F1: 0.8855721393034826
MCC: 0.7483198045536168

FULL Outer Fold 3
   Inner fold: 1
   Inner fold: 2
   Inner fold: 3
Selected threshold: 0.4400000000000003
Accuracy: 0.8369565217391305
F1: 0.8571428571428571
MCC: 0.6691100232489893

FULL Outer Fold 4
   Inner fold: 1
   Inner fold: 2
   I

,Configuration,Accuracy_mean,Accuracy_sd,Balanced_Accuracy_mean,Balanced_Accuracy_sd,Precision_mean,Precision_sd,Recall_mean,Recall_sd,F1_mean,F1_sd,MCC_mean,MCC_sd,ROC_AUC_mean,ROC_AUC_sd,PR_AUC_mean,PR_AUC_sd
0,FULL,0.861689,0.020298,0.858289,0.024071,0.866211,0.038320,0.889750,0.018961,0.877155,0.014526,0.720797,0.041289,0.915257,0.016774,0.919630,0.016589
1,NO_2STAGE,0.855156,0.026031,0.850084,0.027573,0.850360,0.033173,0.897729,0.033422,0.872830,0.022337,0.707651,0.052318,0.915937,0.016688,0.913033,0.015131
2,NO_AFB,0.850790,0.031038,0.847060,0.032021,0.854323,0.035227,0.881926,0.035894,0.867429,0.027166,0.698395,0.063571,0.912283,0.018756,0.916125,0.017562
3,NO_INTERACTION,0.852994,0.027722,0.846171,0.032075,0.841041,0.044624,0.909416,0.024592,0.873016,0.020522,0.704470,0.054094,0.919463,0.016823,0.919007,0.020525
4,NO_LR,0.860573,0.020858,0.853570,0.021457,0.843213,0.021109,0.919336,0.022202,0.879498,0.017908,0.718963,0.042673,0.917208,0.015463,0.918271,0.013652
5,NO_META,0.858428,0.018087,0.853484,0.018562,0.853281,0.022454,0.899651,0.029713,0.875473,0.016045,0.713943,0.037519,0.927545,0.010235,0.930860,0.016119
6,NO_SAE,0.863851,0.009910,0.857229,0.011881,0.848323,0.021666,0.919336,0.022266,0.882019,0.007473,0.726002,0.019731,0.916981,0.016698,0.917266,0.024788
7,NO_THRESHOLD,0.857335,0.029694,0.854148,0.031722,0.862899,0.037504,0.883906,0.020960,0.872983,0.024736,0.711213,0.060987,0.916215,0.018421,0.922773,0.008947


,Ablation,Metric,FULL_mean,Ablation_mean,Difference_FULL_minus_Ablation,Paired_t_p,Wilcoxon_p
0,NO_INTERACTION,Accuracy,0.861689,0.852994,0.008696,0.270695,0.3750
1,NO_INTERACTION,F1,0.877155,0.873016,0.004139,0.549542,0.8125
2,NO_INTERACTION,MCC,0.720797,0.704470,0.016327,0.319622,0.3125
3,NO_INTERACTION,ROC_AUC,0.915257,0.919463,-0.004206,0.452268,0.4375
4,NO_INTERACTION,PR_AUC,0.919630,0.919007,0.000623,0.953199,1.0000
5,NO_SAE,Accuracy,0.861689,0.863851,-0.002162,0.766462,1.0000
6,NO_SAE,F1,0.877155,0.882019,-0.004864,0.351701,0.3125
7,NO_SAE,MCC,0.720797,0.726002,-0.005205,0.718489,0.6250
8,NO_SAE,ROC_AUC,0.915257,0.916981,-0.001724,0.740016,1.0000
9,NO_SAE,PR_AUC,0.919630,0.917266,0.002364,0.822185,0.8125



T4 latency result:
Mean_ms_per_sample : 65.63260496662527
Median_ms_per_sample : 64.33395399972142
SD_ms_per_sample : 3.6063247356863055
95CI_Low : 64.28598117917747
95CI_High : 66.97922875407306
Minimum_ms_per_sample : 61.810604999664065
Maximum_ms_per_sample : 75.38995499999146
Repeats : 30
Warmup : 10
Batch_Size : 1
Parameter count: 35769
Serialized model size (MB): 0.17143535614013672
            Component  Parameters
0  Encoder 120 -> 128       15488
1    Latent 128 -> 64        8256
2        AFB 64 -> 64        4160
3   Decoder 64 -> 120        7800
4  Classifier 64 -> 1          65

Total analytical neural parameters: 35769


,Stage,Complexity,Description
0,Interaction expansion,O(n p^2),Degree-2 interaction-only feature generation
1,SAE encoder,O(n(p*128 + 128*64)),Dense encoder forward computation
2,AFB,O(n*64^2),Adaptive 64-dimensional weighting
3,Decoder,O(n*64*p),Latent reconstruction
4,Classifier,O(n*64),Binary classification output
5,XGBoost training,Approx. O(T*n*p*log n),Tree construction; implementation dependent
6,XGBoost inference,O(T*D),Tree traversal
7,Gradient Boosting meta training,Approx. O(M*n*log n),Meta-ensemble tree construction
8,Gradient Boosting meta inference,O(M*D),Meta-tree traversal



ALL EVIDENCE FILES GENERATED
1. corrected_ablation_outer5_results.csv
2. corrected_ablation_outer5_summary.csv
3. corrected_ablation_statistical_tests.csv
4. corrected_parameter_breakdown.csv
5. corrected_complexity_table.csv
6. T4_latency_results.csv
7. CA_SAE_AFB_Corrected_Evidence.xlsx


In [ ]:
# ============================================================
# FINAL SUPPLEMENTARY ANALYSIS
# PART A — 95% CONFIDENCE INTERVALS
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import t

# ------------------------------------------------------------
# Load corrected fold-level results
# ------------------------------------------------------------

results_file = "corrected_ablation_outer5_results.csv"

ablation_results = pd.read_csv(
    results_file
)

print(
    "Loaded:",
    results_file
)

print(
    ablation_results.head()
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

METRICS = [
    "Accuracy",
    "Balanced_Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "ROC_AUC",
    "PR_AUC"
]


# ------------------------------------------------------------
# Function for 95% CI
# ------------------------------------------------------------

def mean_sd_ci(values):

    values = np.asarray(
        values,
        dtype=float
    )

    n = len(values)

    mean = values.mean()

    sd = values.std(
        ddof=1
    )

    standard_error = (
        sd /
        np.sqrt(n)
    )

    t_critical = t.ppf(
        0.975,
        n - 1
    )

    margin = (
        t_critical *
        standard_error
    )

    lower = (
        mean -
        margin
    )

    upper = (
        mean +
        margin
    )

    return {
        "N": n,
        "Mean": mean,
        "SD": sd,
        "95CI_Lower": lower,
        "95CI_Upper": upper
    }


# ------------------------------------------------------------
# Generate CI table
# ------------------------------------------------------------

ci_rows = []


for configuration in sorted(
    ablation_results[
        "Configuration"
    ].unique()
):

    subset = (
        ablation_results[
            ablation_results[
                "Configuration"
            ]
            ==
            configuration
        ]
    )


    for metric in METRICS:

        statistics = mean_sd_ci(
            subset[metric].values
        )


        ci_rows.append({

            "Configuration":
                configuration,

            "Metric":
                metric,

            "N":
                statistics["N"],

            "Mean":
                statistics["Mean"],

            "SD":
                statistics["SD"],

            "95CI_Lower":
                statistics["95CI_Lower"],

            "95CI_Upper":
                statistics["95CI_Upper"],

            "Mean_SD":
                f"{statistics['Mean']:.4f} ± "
                f"{statistics['SD']:.4f}",

            "95CI":
                f"[{statistics['95CI_Lower']:.4f}, "
                f"{statistics['95CI_Upper']:.4f}]"
        })


ci_table = pd.DataFrame(
    ci_rows
)


display(
    ci_table
)


ci_table.to_csv(
    "corrected_ablation_95CI.csv",
    index=False
)

print(
    "Saved: corrected_ablation_95CI.csv"
)

Loaded: corrected_ablation_outer5_results.csv
   Accuracy  Balanced_Accuracy  Precision    Recall        F1       MCC  \
0  0.842391           0.832736   0.817391  0.921569  0.866359  0.683223   
1  0.875000           0.875299   0.898990  0.872549  0.885572  0.748320   
2  0.836957           0.831420   0.833333  0.882353  0.857143  0.669110   
3  0.879781           0.879618   0.898990  0.881188  0.890000  0.757683   
4  0.874317           0.872374   0.882353  0.891089  0.886700  0.745649   

    ROC_AUC    PR_AUC Configuration  Fold  Threshold  
0  0.901961  0.905099          FULL     1      0.400  
1  0.926650  0.920463          FULL     2      0.525  
2  0.893651  0.903649          FULL     3      0.440  
3  0.920852  0.924644          FULL     4      0.625  
4  0.933168  0.944296          FULL     5      0.365  


,Configuration,Metric,N,Mean,SD,95CI_Lower,95CI_Upper,Mean_SD,95CI
0,FULL,Accuracy,5,0.861689,0.020298,0.836486,0.886893,0.8617 ± 0.0203,"[0.8365, 0.8869]"
1,FULL,Balanced_Accuracy,5,0.858289,0.024071,0.828402,0.888177,0.8583 ± 0.0241,"[0.8284, 0.8882]"
2,FULL,Precision,5,0.866211,0.038320,0.818631,0.913792,0.8662 ± 0.0383,"[0.8186, 0.9138]"
3,FULL,Recall,5,0.889750,0.018961,0.866206,0.913293,0.8897 ± 0.0190,"[0.8662, 0.9133]"
4,FULL,F1,5,0.877155,0.014526,0.859119,0.895191,0.8772 ± 0.0145,"[0.8591, 0.8952]"
...,...,...,...,...,...,...,...,...,...
59,NO_THRESHOLD,Recall,5,0.883906,0.020960,0.857881,0.909931,0.8839 ± 0.0210,"[0.8579, 0.9099]"
60,NO_THRESHOLD,F1,5,0.872983,0.024736,0.842269,0.903697,0.8730 ± 0.0247,"[0.8423, 0.9037]"
61,NO_THRESHOLD,MCC,5,0.711213,0.060987,0.635488,0.786939,0.7112 ± 0.0610,"[0.6355, 0.7869]"
62,NO_THRESHOLD,ROC_AUC,5,0.916215,0.018421,0.893342,0.939088,0.9162 ± 0.0184,"[0.8933, 0.9391]"


Saved: corrected_ablation_95CI.csv


In [ ]:
# ============================================================
# PART B — FULL VS ABLATION EFFECTS
# ============================================================

full = (
    ablation_results[
        ablation_results[
            "Configuration"
        ]
        ==
        "FULL"
    ]
    .sort_values("Fold")
    .reset_index(drop=True)
)


effect_rows = []


for configuration in sorted(
    ablation_results[
        "Configuration"
    ].unique()
):

    if configuration == "FULL":
        continue


    ablation = (
        ablation_results[
            ablation_results[
                "Configuration"
            ]
            ==
            configuration
        ]
        .sort_values("Fold")
        .reset_index(drop=True)
    )


    for metric in METRICS:

        full_values = (
            full[
                metric
            ].values
        )


        ablation_values = (
            ablation[
                metric
            ].values
        )


        difference = (
            full_values.mean()
            -
            ablation_values.mean()
        )


        relative_percent = (

            difference
            /
            abs(
                full_values.mean()
            )

        ) * 100


        effect_rows.append({

            "Ablation":
                configuration,

            "Metric":
                metric,

            "FULL_Mean":
                full_values.mean(),

            "Ablation_Mean":
                ablation_values.mean(),

            "FULL_minus_Ablation":
                difference,

            "Relative_Difference_Percent":
                relative_percent
        })


effect_table = pd.DataFrame(
    effect_rows
)


display(
    effect_table
)


effect_table.to_csv(
    "ablation_effect_vs_full.csv",
    index=False
)

,Ablation,Metric,FULL_Mean,Ablation_Mean,FULL_minus_Ablation,Relative_Difference_Percent
0,NO_2STAGE,Accuracy,0.861689,0.855156,0.006534,0.758234
1,NO_2STAGE,Balanced_Accuracy,0.858289,0.850084,0.008206,0.956042
2,NO_2STAGE,Precision,0.866211,0.850360,0.015851,1.829935
3,NO_2STAGE,Recall,0.889750,0.897729,-0.007979,-0.896773
4,NO_2STAGE,F1,0.877155,0.872830,0.004325,0.493033
5,NO_2STAGE,MCC,0.720797,0.707651,0.013146,1.823846
6,NO_2STAGE,ROC_AUC,0.915257,0.915937,-0.000680,-0.074330
7,NO_2STAGE,PR_AUC,0.919630,0.913033,0.006597,0.717385
8,NO_AFB,Accuracy,0.861689,0.850790,0.010899,1.264872
9,NO_AFB,Balanced_Accuracy,0.858289,0.847060,0.011229,1.308292


In [ ]:
# ============================================================
# PART C — TRUE END-TO-END T4 INFERENCE LATENCY
# ============================================================

import time
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    PolynomialFeatures,
    StandardScaler
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier


# ============================================================
# STEP 1 — Prepare the COMPLETE DATASET
# ============================================================

deployment_df = pd.read_csv(
    filename
)

TARGET = "HeartDisease"


X_deploy = (
    deployment_df
    .drop(columns=[TARGET])
    .copy()
)


y_deploy = (
    deployment_df[TARGET]
    .astype(int)
    .values
)


# One-hot encoding
X_deploy = pd.get_dummies(
    X_deploy,
    drop_first=True
)


X_deploy = X_deploy.replace(
    [np.inf, -np.inf],
    np.nan
)


print(
    "Deployment feature shape:",
    X_deploy.shape
)


# ============================================================
# STEP 2 — FIT PREPROCESSING ON COMPLETE TRAINING DATA
# ============================================================

deployment_imputer = (
    SimpleImputer(
        strategy="median"
    )
)


X_imp = (
    deployment_imputer.fit_transform(
        X_deploy
    )
)


deployment_poly = (
    PolynomialFeatures(
        degree=2,
        interaction_only=True,
        include_bias=False
    )
)


X_poly = (
    deployment_poly.fit_transform(
        X_imp
    )
)


deployment_scaler = (
    StandardScaler()
)


X_scaled = (
    deployment_scaler.fit_transform(
        X_poly
    )
)


X_scaled = X_scaled.astype(
    np.float32
)


print(
    "Expanded feature dimension:",
    X_scaled.shape[1]
)


# ============================================================
# STEP 3 — TRAIN SAE + AFB
# ============================================================

deployment_neural_model, deployment_encoder = (
    train_neural_component(

        X_scaled,

        y_deploy,

        X_scaled,

        y_deploy,

        use_sae=True,

        use_afb=True
    )
)


# ============================================================
# STEP 4 — LATENT REPRESENTATION
# ============================================================

Z_deploy = (
    deployment_encoder.predict(
        X_scaled,
        verbose=0
    )
)


print(
    "Latent dimension:",
    Z_deploy.shape[1]
)


# ============================================================
# STEP 5 — TRAIN LR
# ============================================================

deployment_lr = (
    train_logistic_regression(

        Z_deploy,

        y_deploy,

        SEED
    )
)


# ============================================================
# STEP 6 — TRAIN XGBOOST
# ============================================================

deployment_xgb = (
    train_xgboost(

        X_scaled,

        y_deploy,

        SEED
    )
)


# ============================================================
# STEP 7 — GENERATE OOF PREDICTIONS FOR META LEARNER
# ============================================================
#
# This is required because the final meta learner should not
# be trained on in-sample base-model predictions.
#
# The following function creates OOF predictions on the
# complete development dataset.
# ============================================================

deployment_cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=SEED
)


oof_lr_deploy = np.zeros(
    len(y_deploy)
)


oof_xgb_deploy = np.zeros(
    len(y_deploy)
)


for fold, (
    train_idx,
    validation_idx
) in enumerate(

    deployment_cv.split(
        X_deploy,
        y_deploy
    ),

    start=1
):

    print(
        "Meta OOF fold:",
        fold
    )


    X_train_raw = (
        X_deploy.iloc[
            train_idx
        ]
    )


    X_validation_raw = (
        X_deploy.iloc[
            validation_idx
        ]
    )


    y_train = (
        y_deploy[
            train_idx
        ]
    )


    # --------------------------------------------------------
    # Fold-local imputation
    # --------------------------------------------------------

    imp = SimpleImputer(
        strategy="median"
    )


    A = imp.fit_transform(
        X_train_raw
    )


    B = imp.transform(
        X_validation_raw
    )


    # --------------------------------------------------------
    # Fold-local interactions
    # --------------------------------------------------------

    poly = PolynomialFeatures(

        degree=2,

        interaction_only=True,

        include_bias=False
    )


    A = poly.fit_transform(
        A
    )


    B = poly.transform(
        B
    )


    # --------------------------------------------------------
    # Fold-local scaling
    # --------------------------------------------------------

    scaler = StandardScaler()


    A = scaler.fit_transform(
        A
    )


    B = scaler.transform(
        B
    )


    A = A.astype(
        np.float32
    )


    B = B.astype(
        np.float32
    )


    # --------------------------------------------------------
    # SAE
    # --------------------------------------------------------

    fold_neural, fold_encoder = (
        train_neural_component(

            A,

            y_train,

            B,

            y_deploy[
                validation_idx
            ],

            use_sae=True,

            use_afb=True
        )
    )


    # --------------------------------------------------------
    # Latent
    # --------------------------------------------------------

    ZA = (
        fold_encoder.predict(
            A,
            verbose=0
        )
    )


    ZB = (
        fold_encoder.predict(
            B,
            verbose=0
        )
    )


    # --------------------------------------------------------
    # LR
    # --------------------------------------------------------

    fold_lr = (
        train_logistic_regression(

            ZA,

            y_train,

            SEED + fold
        )
    )


    oof_lr_deploy[
        validation_idx
    ] = (
        fold_lr.predict_proba(
            ZB
        )[:,1]
    )


    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    fold_xgb = (
        train_xgboost(

            A,

            y_train,

            SEED + fold
        )
    )


    oof_xgb_deploy[
        validation_idx
    ] = (
        fold_xgb.predict_proba(
            B
        )[:,1]
    )


    del fold_neural
    del fold_encoder
    del fold_lr
    del fold_xgb

    gc.collect()

    tf.keras.backend.clear_session()


# ============================================================
# STEP 8 — TRAIN META LEARNER ON OOF PREDICTIONS
# ============================================================

deployment_meta = (
    GradientBoostingClassifier(

        n_estimators=200,

        learning_rate=0.05,

        max_depth=2,

        random_state=SEED
    )
)


deployment_meta.fit(

    np.column_stack(
        [
            oof_lr_deploy,

            oof_xgb_deploy
        ]
    ),

    y_deploy
)


# ============================================================
# STEP 9 — VALIDATION-BASED THRESHOLD
# ============================================================

oof_meta_probability = (
    deployment_meta.predict_proba(

        np.column_stack(
            [
                oof_lr_deploy,

                oof_xgb_deploy
            ]
        )

    )[:,1]
)


deployment_threshold = (
    optimize_threshold(

        y_deploy,

        oof_meta_probability
    )
)


print(
    "Deployment threshold:",
    deployment_threshold
)


# ============================================================
# STEP 10 — END-TO-END PREDICTION FUNCTION
# ============================================================

def end_to_end_predict(
    raw_dataframe
):

    start = time.perf_counter()


    # --------------------------------------------------------
    # Raw preprocessing
    # --------------------------------------------------------

    X_raw = raw_dataframe.copy()


    X_raw = X_raw.replace(
        [np.inf, -np.inf],
        np.nan
    )


    X_raw = pd.get_dummies(
        X_raw,
        drop_first=True
    )


    # Align columns with deployment training data
    X_raw = X_raw.reindex(
        columns=X_deploy.columns,
        fill_value=0
    )


    # --------------------------------------------------------
    # Imputation
    # --------------------------------------------------------

    X_raw = (
        deployment_imputer.transform(
            X_raw
        )
    )


    # --------------------------------------------------------
    # Interaction features
    # --------------------------------------------------------

    X_raw = (
        deployment_poly.transform(
            X_raw
        )
    )


    # --------------------------------------------------------
    # Scaling
    # --------------------------------------------------------

    X_raw = (
        deployment_scaler.transform(
            X_raw
        )
    )


    X_raw = X_raw.astype(
        np.float32
    )


    # --------------------------------------------------------
    # SAE + AFB
    # --------------------------------------------------------

    Z = (
        deployment_encoder.predict(
            X_raw,
            verbose=0
        )
    )


    # --------------------------------------------------------
    # LR
    # --------------------------------------------------------

    p_lr = (
        deployment_lr.predict_proba(
            Z
        )[:,1]
    )


    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    p_xgb = (
        deployment_xgb.predict_proba(
            X_raw
        )[:,1]
    )


    # --------------------------------------------------------
    # Meta learner
    # --------------------------------------------------------

    p_meta = (
        deployment_meta.predict_proba(

            np.column_stack(
                [
                    p_lr,

                    p_xgb
                ]
            )

        )[:,1]
    )


    # --------------------------------------------------------
    # Decision
    # --------------------------------------------------------

    prediction = (
        p_meta >=
        deployment_threshold
    ).astype(int)


    end = time.perf_counter()


    elapsed_ms = (
        end -
        start
    ) * 1000


    return (
        prediction,
        p_meta,
        elapsed_ms
    )


# ============================================================
# STEP 11 — SINGLE-SAMPLE T4 LATENCY
# ============================================================

# Use one real record from the dataset.
sample = (
    deployment_df
    .drop(columns=[TARGET])
    .iloc[[0]]
)


# Warm-up
for _ in range(10):

    end_to_end_predict(
        sample
    )


# ------------------------------------------------------------
# Timed measurements
# ------------------------------------------------------------

latencies = []


for i in range(30):

    tf.keras.backend.clear_session()


    start = time.perf_counter()


    prediction, probability, elapsed = (
        end_to_end_predict(
            sample
        )
    )


    end = time.perf_counter()


    latency_ms = (
        end -
        start
    ) * 1000


    latencies.append(
        latency_ms
    )


latencies = np.asarray(
    latencies
)


# ============================================================
# STEP 12 — END-TO-END LATENCY STATISTICS
# ============================================================

mean_latency = (
    latencies.mean()
)


median_latency = (
    np.median(
        latencies
    )
)


sd_latency = (
    latencies.std(
        ddof=1
    )
)


t_critical = t.ppf(
    0.975,
    len(latencies)-1
)


margin = (

    t_critical
    *
    sd_latency
    /
    np.sqrt(
        len(latencies)
    )
)


ci_low = (
    mean_latency -
    margin
)


ci_high = (
    mean_latency +
    margin
)


end_to_end_latency = pd.DataFrame({

    "Mean_ms_per_sample":
        [mean_latency],

    "Median_ms_per_sample":
        [median_latency],

    "SD_ms_per_sample":
        [sd_latency],

    "95CI_Lower":
        [ci_low],

    "95CI_Upper":
        [ci_high],

    "Minimum_ms":
        [latencies.min()],

    "Maximum_ms":
        [latencies.max()],

    "Warmup":
        [10],

    "Timed_Repetitions":
        [30],

    "Batch_Size":
        [1],

    "Threshold":
        [deployment_threshold]

})


display(
    end_to_end_latency
)


end_to_end_latency.to_csv(
    "T4_END_TO_END_LATENCY.csv",
    index=False
)


print(
    "\nEND-TO-END T4 LATENCY:"
)

print(
    f"{mean_latency:.4f} ms/sample"
)

print(
    f"95% CI: "
    f"[{ci_low:.4f}, {ci_high:.4f}]"
)

Deployment feature shape: (918, 15)
Expanded feature dimension: 120
Latent dimension: 64
Meta OOF fold: 1
Meta OOF fold: 2
Meta OOF fold: 3
Meta OOF fold: 4
Meta OOF fold: 5
Deployment threshold: 0.3550000000000002


,Mean_ms_per_sample,Median_ms_per_sample,SD_ms_per_sample,95CI_Lower,95CI_Upper,Minimum_ms,Maximum_ms,Warmup,Timed_Repetitions,Batch_Size,Threshold
0,125.846418,102.597382,93.547232,90.915308,160.777529,96.803207,613.33203,10,30,1,0.355



END-TO-END T4 LATENCY:
125.8464 ms/sample
95% CI: [90.9153, 160.7775]


In [ ]:
# ============================================================
# PART D — 50-SEED VS NESTED/OUTER-CV COMPARISON
# ============================================================

import os
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Change this filename if your original 50-seed result file
# has a different name.
# ------------------------------------------------------------

FIFTY_SEED_FILE = "original_50_seed_results.csv"


if os.path.exists(FIFTY_SEED_FILE):

    old_results = pd.read_csv(
        FIFTY_SEED_FILE
    )

    print(
        "Loaded original 50-seed results."
    )

    display(
        old_results.head()
    )

else:

    print(
        "WARNING:"
    )

    print(
        f"{FIFTY_SEED_FILE} was not found."
    )

    print(
        "Upload your original 50-seed results CSV and rerun this cell."
    )

Loaded original 50-seed results.


,Model,Accuracy,ROC_AUC,F1,MCC,Number_of_Seeds,Evaluation,Seed_Range,Test_Size,Source
0,Logistic Regression,0.8593,0.9243,0.8742,0.7163,50,50 random stratified train/test splits,0-49,0.15,journal_implementation (1)(8).ipynb — executed...
1,Random Forest,0.8732,0.9294,0.8875,0.7446,50,50 random stratified train/test splits,0-49,0.15,journal_implementation (1)(8).ipynb — executed...
2,XGBoost (Standalone),0.8662,0.9235,0.8807,0.7303,50,50 random stratified train/test splits,0-49,0.15,journal_implementation (1)(8).ipynb — executed...
3,MLP (Deep Neural Network),0.8588,0.9231,0.8741,0.7158,50,50 random stratified train/test splits,0-49,0.15,journal_implementation (1)(8).ipynb — executed...
4,Standard AE + LR (No AFB),0.8413,0.9047,0.8573,0.6806,50,50 random stratified train/test splits,0-49,0.15,journal_implementation (1)(8).ipynb — executed...


In [ ]:
# ============================================================
# STANDARDIZE ORIGINAL 50-SEED METRIC NAMES
# ============================================================

def find_column(
    dataframe,
    candidates
):

    for candidate in candidates:

        if candidate in dataframe.columns:

            return candidate

    return None


accuracy_column = find_column(
    old_results,
    [
        "Accuracy",
        "accuracy",
        "accuracy_mean"
    ]
)


f1_column = find_column(
    old_results,
    [
        "F1",
        "F1_score",
        "F1 Score",
        "f1"
    ]
)


mcc_column = find_column(
    old_results,
    [
        "MCC",
        "mcc"
    ]
)


auc_column = find_column(
    old_results,
    [
        "AUC",
        "ROC_AUC",
        "ROC-AUC",
        "roc_auc"
    ]
)


print(
    "Accuracy column:",
    accuracy_column
)

print(
    "F1 column:",
    f1_column
)

print(
    "MCC column:",
    mcc_column
)

print(
    "AUC column:",
    auc_column
)

Accuracy column: Accuracy
F1 column: F1
MCC column: MCC
AUC column: ROC_AUC


In [ ]:
# ============================================================
# 50-SEED SUMMARY
# ============================================================

old_summary = {}


if accuracy_column:

    old_summary[
        "Accuracy"
    ] = old_results[
        accuracy_column
    ].mean()


if f1_column:

    old_summary[
        "F1"
    ] = old_results[
        f1_column
    ].mean()


if mcc_column:

    old_summary[
        "MCC"
    ] = old_results[
        mcc_column
    ].mean()


if auc_column:

    old_summary[
        "ROC_AUC"
    ] = old_results[
        auc_column
    ].mean()


old_summary

In [ ]:
# ============================================================
# 50-SEED VS CORRECTED OUTER-CV
# ============================================================

full_corrected = (
    ablation_results[
        ablation_results[
            "Configuration"
        ]
        ==
        "FULL"
    ]
)


comparison_rows = []


# Re-initialize old_summary to ensure it's available
old_summary = {}

if accuracy_column:

    old_summary[
        "Accuracy"
    ] = old_results[
        accuracy_column
    ].mean()


if f1_column:

    old_summary[
        "F1"
    ] = old_results[
        f1_column
    ].mean()


if mcc_column:

    old_summary[
        "MCC"
    ] = old_results[
        mcc_column
    ].mean()


if auc_column:

    old_summary[
        "ROC_AUC"
    ] = old_results[
        auc_column
    ].mean()


for metric in [

    "Accuracy",

    "F1",

    "MCC",

    "ROC_AUC"

]:

    old_value = (
        old_summary.get(
            metric,
            np.nan
        )
    )


    new_value = (
        full_corrected[
            metric
        ].mean()
    )


    comparison_rows.append({

        "Metric":
            metric,

        "Original_50_seed":
            old_value,

        "Corrected_5Fold":
            new_value,

        "Difference":
            (
                new_value -
                old_value
            ),

        "Relative_change_percent":
            (
                (
                    new_value -
                    old_value
                )
                /
                abs(old_value)
                *
                100
            )
            if not np.isnan(
                old_value
            )
            else np.nan
    })


comparison_50_vs_nested = (
    pd.DataFrame(
        comparison_rows
    )
)


display(
    comparison_50_vs_nested
)


comparison_50_vs_nested.to_csv(
    "50_seed_vs_corrected_CV_comparison.csv",
    index=False
)


,Metric,Original_50_seed,Corrected_5Fold,Difference,Relative_change_percent
0,Accuracy,0.861050,0.861689,0.000639,0.074239
1,F1,0.876017,0.877155,0.001138,0.129920
2,MCC,0.720100,0.720797,0.000697,0.096791
3,ROC_AUC,0.911617,0.915257,0.003640,0.399280


In [ ]:
# ============================================================
# PART E — SAME OUTER-FOLD BASELINE EVALUATION
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


BASELINE_MODELS = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=250,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        ),

    "XGBoost":
        xgb.XGBClassifier(
            n_estimators=250,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            reg_alpha=0.1,
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1
        ),

    "MLP":
        MLPClassifier(
            hidden_layer_sizes=(128,64),
            activation="relu",
            solver="adam",
            alpha=0.0001,
            batch_size=32,
            learning_rate_init=0.0005,
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.20,
            n_iter_no_change=10,
            random_state=SEED
        )
}


baseline_rows = []


outer_cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=SEED
)


for model_name, base_model in (
    BASELINE_MODELS.items()
):

    print(
        "\n===================================="
    )

    print(
        model_name
    )

    print(
        "===================================="
    )


    for fold, (
        train_idx,
        test_idx
    ) in enumerate(

        outer_cv.split(
            X_df,
            y
        ),

        start=1
    ):


        X_train_raw = (
            X_df.iloc[
                train_idx
            ]
        )


        X_test_raw = (
            X_df.iloc[
                test_idx
            ]
        )


        y_train = (
            y[
                train_idx
            ]
        )


        y_test = (
            y[
                test_idx
            ]
        )


        # ----------------------------------------------------
        # Fold-local preprocessing
        # ----------------------------------------------------

        (
            X_train,
            _,
            X_test,
            _,
            _,
            _
        ) = preprocess_fold(

            X_train_raw,

            X_train_raw,

            X_test_raw,

            use_interactions=True
        )


        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        model = (
            base_model
        )


        model.fit(

            X_train,

            y_train
        )


        # ----------------------------------------------------
        # Probability
        # ----------------------------------------------------

        probability = (
            model.predict_proba(
                X_test
            )[:,1]
        )


        prediction = (
            probability >= 0.5
        ).astype(int)


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        baseline_rows.append({

            "Model":
                model_name,

            "Fold":
                fold,

            "Accuracy":
                accuracy_score(
                    y_test,
                    prediction
                ),

            "Balanced_Accuracy":
                balanced_accuracy_score(
                    y_test,
                    prediction
                ),

            "Precision":
                precision_score(
                    y_test,
                    prediction,
                    zero_division=0
                ),

            "Recall":
                recall_score(
                    y_test,
                    prediction,
                    zero_division=0
                ),

            "F1":
                f1_score(
                    y_test,
                    prediction,
                    zero_division=0
                ),

            "MCC":
                matthews_corrcoef(
                    y_test,
                    prediction
                ),

            "ROC_AUC":
                roc_auc_score(
                    y_test,
                    probability
                ),

            "PR_AUC":
                average_precision_score(
                    y_test,
                    probability
                )
        })


        gc.collect()


baseline_results = pd.DataFrame(
    baseline_rows
)


baseline_results.to_csv(
    "same_fold_baseline_results.csv",
    index=False
)


display(
    baseline_results
)


Logistic Regression

Random Forest

XGBoost

MLP


,Model,Fold,Accuracy,Balanced_Accuracy,Precision,Recall,F1,MCC,ROC_AUC,PR_AUC
0,Logistic Regression,1,0.842391,0.839909,0.854369,0.862745,0.858537,0.680673,0.886538,0.881087
1,Logistic Regression,2,0.826087,0.822812,0.836538,0.852941,0.844660,0.647329,0.918580,0.930537
2,Logistic Regression,3,0.842391,0.842300,0.868687,0.843137,0.855721,0.682523,0.892396,0.889129
3,Logistic Regression,4,0.879781,0.877324,0.883495,0.900990,0.892157,0.756569,0.936972,0.945637
4,Logistic Regression,5,0.857923,0.856375,0.871287,0.871287,0.871287,0.712751,0.931055,0.934095
5,Random Forest,1,0.858696,0.851028,0.839286,0.921569,0.878505,0.714996,0.914873,0.927844
6,Random Forest,2,0.875000,0.871712,0.876190,0.901961,0.888889,0.746510,0.945182,0.960021
7,Random Forest,3,0.831522,0.826518,0.831776,0.872549,0.851675,0.657972,0.906265,0.911582
8,Random Forest,4,0.890710,0.886078,0.878505,0.930693,0.903846,0.779245,0.935825,0.937767
9,Random Forest,5,0.890710,0.890667,0.909091,0.891089,0.900000,0.779734,0.929667,0.910427


In [ ]:
# ============================================================
# BASELINE SUMMARY + 95% CI
# ============================================================

baseline_ci_rows = []


for model_name in (
    baseline_results["Model"]
    .unique()
):

    subset = (
        baseline_results[
            baseline_results[
                "Model"
            ]
            ==
            model_name
        ]
    )


    for metric in METRICS:

        statistics = mean_sd_ci(
            subset[
                metric
            ].values
        )


        baseline_ci_rows.append({

            "Model":
                model_name,

            "Metric":
                metric,

            "Mean":
                statistics["Mean"],

            "SD":
                statistics["SD"],

            "95CI_Lower":
                statistics["95CI_Lower"],

            "95CI_Upper":
                statistics["95CI_Upper"]
        })


baseline_ci = pd.DataFrame(
    baseline_ci_rows
)


display(
    baseline_ci
)


baseline_ci.to_csv(
    "same_fold_baseline_95CI.csv",
    index=False
)

,Model,Metric,Mean,SD,95CI_Lower,95CI_Upper
0,Logistic Regression,Accuracy,0.849715,0.020229,0.824597,0.874833
1,Logistic Regression,Balanced_Accuracy,0.847744,0.020385,0.822433,0.873055
2,Logistic Regression,Precision,0.862875,0.017998,0.840528,0.885223
3,Logistic Regression,Recall,0.866220,0.022113,0.838764,0.893676
4,Logistic Regression,F1,0.864472,0.018147,0.841940,0.887005
5,Logistic Regression,MCC,0.695969,0.041032,0.645021,0.746918
6,Logistic Regression,ROC_AUC,0.913108,0.022674,0.884954,0.941262
7,Logistic Regression,PR_AUC,0.916097,0.028974,0.880120,0.952073
8,Random Forest,Accuracy,0.869328,0.024945,0.838355,0.900301
9,Random Forest,Balanced_Accuracy,0.865201,0.026570,0.832209,0.898192


In [ ]:
# ============================================================
# PART F — FINAL REBUTTAL EVIDENCE WORKBOOK
# ============================================================

with pd.ExcelWriter(

    "FINAL_CA_SAE_AFB_REVIEWER_EVIDENCE.xlsx",

    engine="openpyxl"

) as writer:


    # ----------------------------------------
    # Ablation
    # ----------------------------------------

    ablation_results.to_excel(

        writer,

        index=False,

        sheet_name="Ablation_Folds"
    )


    summary.to_excel(

        writer,

        index=False,

        sheet_name="Ablation_Summary"
    )


    ci_table.to_excel(

        writer,

        index=False,

        sheet_name="Ablation_95CI"
    )


    effect_table.to_excel(

        writer,

        index=False,

        sheet_name="Ablation_Effects"
    )


    ablation_statistics.to_excel(

        writer,

        index=False,

        sheet_name="Ablation_Statistics"
    )


    # ----------------------------------------
    # T4
    # ----------------------------------------

    end_to_end_latency.to_excel(

        writer,

        index=False,

        sheet_name="T4_EndToEnd"
    )


    # ----------------------------------------
    # Baselines
    # ----------------------------------------

    baseline_results.to_excel(

        writer,

        index=False,

        sheet_name="Baseline_Folds"
    )


    baseline_ci.to_excel(

        writer,

        index=False,

        sheet_name="Baseline_95CI"
    )


    # ----------------------------------------
    # 50-seed comparison
    # ----------------------------------------

    if "comparison_50_vs_nested" in globals():

        comparison_50_vs_nested.to_excel(

            writer,

            index=False,

            sheet_name="50Seed_vs_CV"
        )


    # ----------------------------------------
    # Complexity
    # ----------------------------------------

    parameter_breakdown.to_excel(

        writer,

        index=False,

        sheet_name="Parameters"
    )


    complexity_table.to_excel(

        writer,

        index=False,

        sheet_name="Complexity"
    )


print(
    "============================================"
)

print(
    "FINAL REVIEWER EVIDENCE WORKBOOK CREATED"
)

print(
    "============================================"
)

print(
    "FINAL_CA_SAE_AFB_REVIEWER_EVIDENCE.xlsx"
)

FINAL REVIEWER EVIDENCE WORKBOOK CREATED
FINAL_CA_SAE_AFB_REVIEWER_EVIDENCE.xlsx


In [ ]:
# ============================================================
# FINAL T4 END-TO-END LATENCY EXPERIMENT
# ============================================================
#
# Reviewer 3:
# "Document the exact hardware environment (CPU/GPU specs).
# Add a table reporting the per-sample inference latency in
# milliseconds; a model intended for clinical decision support
# must justify its operational speed."
#
# This experiment measures the COMPLETE inference pipeline:
#
# Raw input
#     ↓
# Column alignment
#     ↓
# Missing-value imputation
#     ↓
# Interaction generation
#     ↓
# Scaling
#     ↓
# SAE + AFB
#     ↓
# LR
#     ↓
# XGBoost
#     ↓
# Gradient Boosting meta learner
#     ↓
# Final prediction
#
# IMPORTANT:
# Run this cell on Google Colab with NVIDIA T4 selected.
# ============================================================


import os
import gc
import time
import platform
import subprocess

import numpy as np
import pandas as pd

from scipy.stats import t


# ============================================================
# 1. HARDWARE INFORMATION
# ============================================================

print("=" * 70)
print("HARDWARE ENVIRONMENT")
print("=" * 70)

print("\nPython:")
print(platform.python_version())

print("\nOperating system:")
print(platform.platform())

print("\nCPU:")
try:
    print(
        subprocess.check_output(
            "lscpu | grep -E 'Model name|CPU\\(s\\)'",
            shell=True,
            text=True
        )
    )
except Exception as e:
    print(e)

print("\nRAM:")
try:
    print(
        subprocess.check_output(
            "free -h",
            shell=True,
            text=True
        )
    )
except Exception as e:
    print(e)

print("\nGPU:")
try:
    print(
        subprocess.check_output(
            "nvidia-smi",
            shell=True,
            text=True
        )
    )
except Exception as e:
    print(e)

print("\nGPU summary:")
try:
    print(
        subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,"
                "driver_version,compute_cap",
                "--format=csv,noheader"
            ],
            text=True
        )
    )
except Exception as e:
    print(e)


# ============================================================
# 2. GPU SYNCHRONIZATION
# ============================================================

def synchronize_gpu():

    """
    Force synchronization before and after timing.

    This is important because GPU operations can be
    asynchronous.
    """

    try:

        import torch

        if torch.cuda.is_available():

            torch.cuda.synchronize()

    except Exception:

        pass

    try:

        # TensorFlow synchronization
        tf.config.experimental.async_wait()

    except Exception:

        pass


# ============================================================
# 3. PREPARE ONE REAL CLINICAL SAMPLE
# ============================================================

# Use one real observation from the original dataset.

raw_sample = (
    df
    .drop(columns=[TARGET])
    .iloc[[0]]
    .copy()
)


print("\nRaw sample:")
display(raw_sample)


# ============================================================
# 4. CREATE END-TO-END PREDICTION FUNCTION
# ============================================================
#
# This function uses the trained objects already created
# in your notebook:
#
# deployment_imputer
# deployment_poly
# deployment_scaler
# deployment_encoder
# deployment_lr
# deployment_xgb
# deployment_meta
# deployment_threshold
# X_deploy
#
# If your variable names differ, change them here only.
# ============================================================


def complete_clinical_inference(
    raw_dataframe
):

    # --------------------------------------------------------
    # START TOTAL TIMER
    # --------------------------------------------------------

    total_start = time.perf_counter()


    # ========================================================
    # STAGE 1 — INPUT PREPARATION
    # ========================================================

    preprocessing_start = time.perf_counter()


    X_raw = raw_dataframe.copy()


    # Replace invalid values

    X_raw = X_raw.replace(
        [np.inf, -np.inf],
        np.nan
    )


    # One-hot encoding

    X_raw = pd.get_dummies(
        X_raw,
        drop_first=True
    )


    # Make sure the columns are identical to training

    X_raw = X_raw.reindex(
        columns=X_deploy.columns,
        fill_value=0
    )


    # --------------------------------------------------------
    # Imputation
    # --------------------------------------------------------

    X_raw = deployment_imputer.transform(
        X_raw
    )


    # --------------------------------------------------------
    # Interaction features
    # --------------------------------------------------------

    X_raw = deployment_poly.transform(
        X_raw
    )


    # --------------------------------------------------------
    # Scaling
    # --------------------------------------------------------

    X_raw = deployment_scaler.transform(
        X_raw
    )


    X_raw = X_raw.astype(
        np.float32
    )


    preprocessing_end = time.perf_counter()


    preprocessing_ms = (
        preprocessing_end
        -
        preprocessing_start
    ) * 1000


    # ========================================================
    # STAGE 2 — SAE + AFB
    # ========================================================

    neural_start = time.perf_counter()


    Z = deployment_encoder.predict(
        X_raw,
        verbose=0
    )


    synchronize_gpu()


    neural_end = time.perf_counter()


    neural_ms = (
        neural_end
        -
        neural_start
    ) * 1000


    # ========================================================
    # STAGE 3 — LOGISTIC REGRESSION
    # ========================================================

    lr_start = time.perf_counter()


    p_lr = deployment_lr.predict_proba(
        Z
    )[:, 1]


    lr_end = time.perf_counter()


    lr_ms = (
        lr_end
        -
        lr_start
    ) * 1000


    # ========================================================
    # STAGE 4 — XGBOOST
    # ========================================================

    xgb_start = time.perf_counter()


    p_xgb = deployment_xgb.predict_proba(
        X_raw
    )[:, 1]


    xgb_end = time.perf_counter()


    xgb_ms = (
        xgb_end
        -
        xgb_start
    ) * 1000


    # ========================================================
    # STAGE 5 — GRADIENT BOOSTING META LEARNER
    # ========================================================

    meta_start = time.perf_counter()


    meta_input = np.column_stack(
        [
            p_lr,
            p_xgb
        ]
    )


    p_meta = deployment_meta.predict_proba(
        meta_input
    )[:, 1]


    meta_end = time.perf_counter()


    meta_ms = (
        meta_end
        -
        meta_start
    ) * 1000


    # ========================================================
    # STAGE 6 — FINAL DECISION
    # ========================================================

    decision_start = time.perf_counter()


    prediction = (
        p_meta >= deployment_threshold
    ).astype(int)


    decision_end = time.perf_counter()


    decision_ms = (
        decision_end
        -
        decision_start
    ) * 1000


    # ========================================================
    # TOTAL
    # ========================================================

    total_end = time.perf_counter()


    total_ms = (
        total_end
        -
        total_start
    ) * 1000


    return {

        "prediction":
            int(prediction[0]),

        "probability":
            float(p_meta[0]),

        "preprocessing_ms":
            preprocessing_ms,

        "SAE_AFB_ms":
            neural_ms,

        "LR_ms":
            lr_ms,

        "XGBoost_ms":
            xgb_ms,

        "MetaLearner_ms":
            meta_ms,

        "Decision_ms":
            decision_ms,

        "total_ms":
            total_ms
    }


# ============================================================
# 5. WARM-UP
# ============================================================
#
# First executions can include:
# - TensorFlow graph initialization
# - CUDA initialization
# - memory allocation
# - kernel compilation
#
# Therefore they MUST NOT be included in the final latency.
# ============================================================

print("\n" + "=" * 70)
print("WARM-UP")
print("=" * 70)


WARMUP_RUNS = 50


for i in range(
    WARMUP_RUNS
):

    synchronize_gpu()

    complete_clinical_inference(
        raw_sample
    )

    synchronize_gpu()


print(
    f"Completed {WARMUP_RUNS} warm-up runs."
)


# ============================================================
# 6. TIMED EXPERIMENT
# ============================================================

print("\n" + "=" * 70)
print("TIMED EXPERIMENT")
print("=" * 70)


TIMED_RUNS = 200


measurements = []


for i in range(
    TIMED_RUNS
):

    synchronize_gpu()


    result = (
        complete_clinical_inference(
            raw_sample
        )
    )


    synchronize_gpu()


    measurements.append(
        result
    )


    if (
        (i + 1) % 25
        == 0
    ):

        print(
            f"Completed {i+1}/{TIMED_RUNS}"
        )


# ============================================================
# 7. CONVERT TO DATAFRAME
# ============================================================

latency_df = pd.DataFrame(
    measurements
)


print("\nLatency measurements:")
display(
    latency_df.head()
)


# ============================================================
# 8. REMOVE EXTREME RUNTIME OUTLIERS
# ============================================================
#
# IMPORTANT:
#
# We do NOT silently delete observations.
#
# First report raw statistics.
#
# Then calculate robust statistics using the 1st and 99th
# percentiles for sensitivity analysis.
# ============================================================

raw_latency = (
    latency_df[
        "total_ms"
    ]
    .astype(float)
    .values
)


# ============================================================
# 9. RAW LATENCY STATISTICS
# ============================================================

def calculate_latency_statistics(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )


    n = len(values)


    mean_value = (
        np.mean(values)
    )


    median_value = (
        np.median(values)
    )


    sd_value = (
        np.std(
            values,
            ddof=1
        )
    )


    standard_error = (
        sd_value
        /
        np.sqrt(n)
    )


    t_critical = (
        t.ppf(
            0.975,
            n - 1
        )
    )


    margin = (
        t_critical
        *
        standard_error
    )


    return {

        "N":
            n,

        "Mean_ms":
            mean_value,

        "Median_ms":
            median_value,

        "SD_ms":
            sd_value,

        "95CI_Lower_ms":
            mean_value - margin,

        "95CI_Upper_ms":
            mean_value + margin,

        "P50_ms":
            np.percentile(
                values,
                50
            ),

        "P95_ms":
            np.percentile(
                values,
                95
            ),

        "P99_ms":
            np.percentile(
                values,
                99
            ),

        "Min_ms":
            np.min(values),

        "Max_ms":
            np.max(values)
    }


raw_statistics = (
    calculate_latency_statistics(
        raw_latency
    )
)


print("\nRAW LATENCY STATISTICS")
print("=" * 70)


for key, value in (
    raw_statistics.items()
):

    print(
        f"{key}: {value:.6f}"
        if isinstance(
            value,
            float
        )
        else
        f"{key}: {value}"
    )


# ============================================================
# 10. ROBUST LATENCY STATISTICS
# ============================================================
#
# This is a sensitivity analysis only.
#
# The raw measurements remain available.
# ============================================================

lower_cutoff = (
    np.percentile(
        raw_latency,
        1
    )
)


upper_cutoff = (
    np.percentile(
        raw_latency,
        99
    )
)


robust_latency = (
    raw_latency[
        (
            raw_latency
            >= lower_cutoff
        )
        &
        (
            raw_latency
            <= upper_cutoff
        )
    ]
)


robust_statistics = (
    calculate_latency_statistics(
        robust_latency
    )
)


print("\nROBUST LATENCY STATISTICS")
print("=" * 70)


print(
    "1st percentile cutoff:",
    lower_cutoff
)


print(
    "99th percentile cutoff:",
    upper_cutoff
)


for key, value in (
    robust_statistics.items()
):

    print(
        f"{key}: {value:.6f}"
        if isinstance(
            value,
            float
        )
        else
        f"{key}: {value}"
    )


# ============================================================
# 11. STAGE-WISE LATENCY
# ============================================================

stage_metrics = [

    "preprocessing_ms",

    "SAE_AFB_ms",

    "LR_ms",

    "XGBoost_ms",

    "MetaLearner_ms",

    "Decision_ms"

]


stage_summary = []


for stage in stage_metrics:

    values = (
        latency_df[
            stage
        ]
        .astype(float)
        .values
    )


    stage_summary.append({

        "Stage":
            stage,

        "Mean_ms":
            np.mean(values),

        "Median_ms":
            np.median(values),

        "SD_ms":
            np.std(
                values,
                ddof=1
            ),

        "P95_ms":
            np.percentile(
                values,
                95
            ),

        "P99_ms":
            np.percentile(
                values,
                99
            ),

        "Min_ms":
            np.min(values),

        "Max_ms":
            np.max(values)
    })


stage_summary_df = (
    pd.DataFrame(
        stage_summary
    )
)


print("\nSTAGE-WISE LATENCY")
print("=" * 70)

display(
    stage_summary_df
)


# ============================================================
# 12. THROUGHPUT
# ============================================================

mean_latency_seconds = (
    raw_statistics[
        "Mean_ms"
    ]
    /
    1000
)


throughput_samples_per_second = (

    1
    /
    mean_latency_seconds
)


print(
    "\nEstimated single-sample throughput:",
    throughput_samples_per_second,
    "samples/second"
)


# ============================================================
# 13. FINAL LATENCY TABLE
# ============================================================

final_latency_table = pd.DataFrame({

    "Metric": [

        "Warm-up runs",

        "Timed runs",

        "Batch size",

        "Mean latency (ms/sample)",

        "Median latency (ms/sample)",

        "SD (ms/sample)",

        "95% CI lower (ms/sample)",

        "95% CI upper (ms/sample)",

        "P50 latency (ms/sample)",

        "P95 latency (ms/sample)",

        "P99 latency (ms/sample)",

        "Minimum latency (ms/sample)",

        "Maximum latency (ms/sample)",

        "Throughput (samples/s)"

    ],

    "Value": [

        WARMUP_RUNS,

        TIMED_RUNS,

        1,

        raw_statistics[
            "Mean_ms"
        ],

        raw_statistics[
            "Median_ms"
        ],

        raw_statistics[
            "SD_ms"
        ],

        raw_statistics[
            "95CI_Lower_ms"
        ],

        raw_statistics[
            "95CI_Upper_ms"
        ],

        raw_statistics[
            "P50_ms"
        ],

        raw_statistics[
            "P95_ms"
        ],

        raw_statistics[
            "P99_ms"
        ],

        raw_statistics[
            "Min_ms"
        ],

        raw_statistics[
            "Max_ms"
        ],

        throughput_samples_per_second

    ]
})


display(
    final_latency_table
)


# ============================================================
# 14. SAVE ALL LATENCY EVIDENCE
# ============================================================

latency_df.to_csv(

    "T4_end_to_end_all_measurements.csv",

    index=False
)


final_latency_table.to_csv(

    "T4_end_to_end_latency_final.csv",

    index=False
)


stage_summary_df.to_csv(

    "T4_stagewise_latency.csv",

    index=False
)


pd.DataFrame(
    [raw_statistics]
).to_csv(

    "T4_latency_raw_statistics.csv",

    index=False
)


pd.DataFrame(
    [robust_statistics]
).to_csv(

    "T4_latency_robust_statistics.csv",

    index=False
)


print("\n" + "=" * 70)
print("FILES GENERATED")
print("=" * 70)


print(
    "1. T4_end_to_end_all_measurements.csv"
)

print(
    "2. T4_end_to_end_latency_final.csv"
)

print(
    "3. T4_stagewise_latency.csv"
)

print(
    "4. T4_latency_raw_statistics.csv"
)

print(
    "5. T4_latency_robust_statistics.csv"
)

HARDWARE ENVIRONMENT

Python:
3.12.13

Operating system:
Linux-6.6.122+-x86_64-with-glibc2.35

CPU:
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
NUMA node0 CPU(s):                       0,1


RAM:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       9.8Gi       942Mi        16Mi       2.0Gi       2.6Gi
Swap:             0B          0B          0B


GPU:
Mon Aug 17 07:40:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-U

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up



WARM-UP
Completed 50 warm-up runs.

TIMED EXPERIMENT
Completed 25/200
Completed 50/200
Completed 75/200
Completed 100/200
Completed 125/200
Completed 150/200
Completed 175/200
Completed 200/200

Latency measurements:


,prediction,probability,preprocessing_ms,SAE_AFB_ms,LR_ms,XGBoost_ms,MetaLearner_ms,Decision_ms,total_ms
0,0,0.201527,19.403074,117.878339,0.480397,2.543354,0.880415,0.008802,141.200867
1,0,0.201527,7.743824,134.236077,0.523644,7.817885,0.919321,0.008660,151.255318
2,0,0.201527,7.725491,136.132476,0.524410,1.927410,0.807507,0.008077,147.131010
3,0,0.201527,9.545677,104.217611,0.499309,11.071102,1.225340,0.010286,126.576081
4,0,0.201527,19.073390,146.506558,0.496857,11.440624,0.894104,0.010676,178.441423



RAW LATENCY STATISTICS
N: 200
Mean_ms: 83.179571
Median_ms: 74.570889
SD_ms: 21.746150
95CI_Lower_ms: 80.147323
95CI_Upper_ms: 86.211819
P50_ms: 74.570889
P95_ms: 139.873329
P99_ms: 160.704551
Min_ms: 70.148478
Max_ms: 178.441423

ROBUST LATENCY STATISTICS
1st percentile cutoff: 70.8713457499016
99th percentile cutoff: 160.70455061967542
N: 196
Mean_ms: 82.388032
Median_ms: 74.570889
SD_ms: 19.903064
95CI_Lower_ms: 79.584253
95CI_Upper_ms: 85.191811
P50_ms: 74.570889
P95_ms: 136.206247
P99_ms: 158.314743
Min_ms: 70.875429
Max_ms: 160.622750

STAGE-WISE LATENCY


,Stage,Mean_ms,Median_ms,SD_ms,P95_ms,P99_ms,Min_ms,Max_ms
0,preprocessing_ms,6.976056,6.181268,2.530203,10.435492,19.288820,5.618641,19.616620
1,SAE_AFB_ms,73.604010,66.382014,18.459618,123.924086,144.308276,62.776229,146.575781
2,LR_ms,0.309819,0.282435,0.067113,0.443421,0.524500,0.241602,0.681567
3,XGBoost_ms,1.483168,0.817854,2.345838,7.882784,11.664805,0.643773,13.017745
4,MetaLearner_ms,0.793829,0.755287,0.162762,1.035850,1.241911,0.647552,2.062863
5,Decision_ms,0.007504,0.006876,0.002821,0.009462,0.022692,0.005702,0.026232



Estimated single-sample throughput: 12.022182753345952 samples/second


,Metric,Value
0,Warm-up runs,50.000000
1,Timed runs,200.000000
2,Batch size,1.000000
3,Mean latency (ms/sample),83.179571
4,Median latency (ms/sample),74.570889
5,SD (ms/sample),21.746150
6,95% CI lower (ms/sample),80.147323
7,95% CI upper (ms/sample),86.211819
8,P50 latency (ms/sample),74.570889
9,P95 latency (ms/sample),139.873329



FILES GENERATED
1. T4_end_to_end_all_measurements.csv
2. T4_end_to_end_latency_final.csv
3. T4_stagewise_latency.csv
4. T4_latency_raw_statistics.csv
5. T4_latency_robust_statistics.csv
